# 06 — Keyboard Widget

The `KeyboardWidget` provides a **musical typing keyboard** (inspired by Logic Pro's layout)
directly inside the notebook.  It maps QWERTY keys to musical notes across two octave
halves — an upper half and a lower half — each independently transposable.

In this notebook we will:
1. Create a standalone keyboard and play notes
2. Connect it to a sequencer for live recording
3. Connect it to samplers with different routing modes

In [ ]:
import math

from IPython.display import display

from nbplay import KeyboardWidget, MixerWidget, NoteComposer, SamplerWidget, SequencerWidget, Session

## Standalone Keyboard

Create a keyboard and display it.  **Click** the widget to focus it, then use
your QWERTY keys to play notes.

| Row | Keys | Role |
|-----|------|------|
| Number row | `2 3 _ 5 6 7 _ 9 0` | Upper sharps |
| QWERTY row | `Q W E R T Y U I O P` | Upper naturals |
| Home row | `A S _ F G H _ K L` | Lower sharps |
| Bottom row | `Z X C V B N M , .` | Lower naturals |

**Controls:** `[ ]` shift upper octave, `; '` shift lower octave,
`- =` velocity down/up (hold to accelerate),
`` ` `` sustain upper, `/` sustain lower, `Space` sustain all.

In [ ]:
kb = KeyboardWidget(upper_octave=3, lower_octave=4)
kb

You can adjust octaves and velocity programmatically too:

In [ ]:
kb.upper_octave = 4
kb.velocity = 80
print(f"Upper octave: {kb.upper_octave}, Lower octave: {kb.lower_octave}, Velocity: {kb.velocity}")

## Connecting to a Sequencer

When a keyboard is connected to a sequencer, the sequencer gains a **REC** button.
With REC armed and the sequencer playing, notes you play on the keyboard are
automatically written to voice 1 at the current step.

You can also **double-click** a step cell in the sequencer grid — instead of
typing a note name, press a key on the keyboard and the note is captured.

In [ ]:
seq = SequencerWidget(length=8, num_voices=2, bpm=100)

# Pre-populate voice 0 with a simple pattern
for i in range(0, 8, 2):
    seq.set_step(i, note=60 + i, velocity=100, voice=0)

# Connect keyboard
kb2 = KeyboardWidget()
kb2.connect_sequencer(seq)
print(f"Sequencer keyboard_connected: {seq.keyboard_connected}")


In [ ]:
display(kb2)
display(seq)

## Connecting to a Sampler

The keyboard can trigger samples loaded in a `SamplerWidget`.  For the audio
routing to work the sampler needs a **session bus** — this is created
automatically when you use a `Session` and call `add_track`.

There are three routing modes:

| Mode | API | Description |
|------|-----|-------------|
| **Whole keyboard** | `connect_sampler(samp)` | Every key triggers the same sampler |
| **Upper/Lower split** | `connect_sampler(samp, zone="upper")` | Only keys in that half trigger this sampler |
| **Per-key** | (planned) | Assign individual keys to different samplers |

Once routing is set up, the sampler's **ADSR envelope** controls
(attack, decay, sustain, release) apply to every note played from the keyboard.

### Single sampler — whole keyboard

Create a `Session` with a sampler track.  `add_track` wires the sampler's
`session_id` and `channel_index` automatically, so the keyboard can find it
on the session bus.  Then `connect_sampler` tells the keyboard to route all
keys through that sampler.

In [ ]:
# Create a sampler with a short synthesised blip sample
samp = SamplerWidget(attack=0.01, decay=0.2, sustain=0.6, release=0.3)
sr = 44100
n_frames = int(sr * 0.25)
data = [math.sin(2 * math.pi * 440 * i / sr) * (1 - i / n_frames) for i in range(n_frames)]
samp.load_sample(data, sample_rate=sr, root_note=60, name="Blip")

# Build a session so the mixer creates the audio bus
session = Session(bpm=120)
seq3 = SequencerWidget(length=8, bpm=120)
session.add_track("Sampler", seq3, samp)

# Connect keyboard to the sampler
kb3 = KeyboardWidget()
kb3.session_id = session._session_id
kb3.connect_sampler(samp, zone="all")

print(f"Sampler session_id: {samp.session_id}")
print(f"Sampler channel_index: {samp.channel_index}")
print(f"Keyboard routing: {kb3.sampler_routing}")

display(kb3, samp, session.mixer)

### Upper/Lower split — two samplers

Route the upper keyboard rows (QWERTY) to one sampler and the lower rows (ZXCV)
to another.  Each sampler gets its own ADSR — try changing the attack on one
while leaving the other fast.

In [ ]:
# Two samplers with different timbres
samp_hi = SamplerWidget(attack=0.01, decay=0.1, sustain=0.8, release=0.2)
samp_hi.load_sample(data, sample_rate=sr, root_note=60, name="Hi Blip")

samp_lo = SamplerWidget(attack=0.3, decay=0.2, sustain=0.5, release=0.5)
samp_lo.load_sample(data, sample_rate=sr, root_note=48, name="Lo Blip")

# Session with two sampler tracks
session2 = Session(bpm=120)
session2.add_track("Hi", SequencerWidget(length=8), samp_hi)
session2.add_track("Lo", SequencerWidget(length=8), samp_lo)

kb4 = KeyboardWidget()
kb4.session_id = session2._session_id
kb4.connect_sampler(samp_hi, zone="upper")
kb4.connect_sampler(samp_lo, zone="lower")

print(f"Sampler routing: {kb4.sampler_routing}")
display(kb4, samp_hi, samp_lo, session2.mixer)

### Disconnect

In [ ]:
kb4.disconnect_sampler(samp_hi)
print(f"Routing after disconnect: {kb4.sampler_routing}")
print(f"Hi sampler keyboard_connected: {samp_hi.keyboard_connected}")